# ML-04 — Search Intelligence Data Contract

This notebook defines the analysis table used for the Week 3 data-contract work and checks the
claims against the full FlyRank warehouse release.

I use **March 2026 as the mid-panel snapshot**. The raw warehouse fact is daily, so the analysis
grain below is one pseudonymized content item for the March 2026 snapshot, with historical
features calculated only from data available before March and the March outcome used as the label.

The warehouse is read through DuckDB from Hugging Face. No warehouse data is written to this repo.

## 0. Setup and authentication

The repository instructions say not to hardcode a Hugging Face token because the repository is
public. This notebook therefore reads `HF_TOKEN` from the environment, Colab Secrets, or a
private prompt.

Before running this notebook, the Hugging Face account used for the token must already have
accepted the `FlyRank/internship-warehouse` access gate.

In [1]:
# Setup: local Hugging Face authentication + DuckDB

from pathlib import Path
from dotenv import load_dotenv
import os
import duckdb
import pandas as pd

HERE = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [HERE, *HERE.parents] if (p / ".env").exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find .env. Create D:\\GodCPP\\fl1\\FLrank1\\.env "
        "with HF_TOKEN=<your token>."
    )

load_dotenv(PROJECT_ROOT / ".env")
HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing from the .env file.")

print("Token loaded:", True)

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(
    "CREATE OR REPLACE SECRET hf "
    f"(TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

# Give remote Hugging Face reads more time to recover from transient SSL errors.
con.execute("SET http_timeout = 120")
con.execute("SET http_retries = 10")
con.execute("SET http_retry_wait_ms = 1000")
con.execute("SET http_retry_backoff = 2")
con.execute("SET http_keep_alive = false")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "content": f"read_parquet('{REL}/dim_content.parquet')",
    "daily_mar": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    "daily_feb": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')",
    "daily_jan": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-01/*.parquet')",
    "daily_dec": f"read_parquet('{REL}/fact_content_daily_performance/month=2025-12/*.parquet')",
    "query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB authentication configured.")
print("Tables:", ", ".join(TABLES.keys()))


Token loaded: True
DuckDB authentication configured.
Tables: clients, content, daily_mar, daily_feb, daily_jan, daily_dec, query_90d


In [2]:
# HTTP settings are configured in the setup cell above.
print("DuckDB HTTP settings configured.")


DuckDB HTTP settings configured.


In [3]:
print("=== PARTITION CHECK ===")

for name in ["daily_dec", "daily_jan", "daily_feb", "daily_mar"]:
    n = con.sql(
        f"SELECT COUNT(*) FROM {TABLES[name]}"
    ).fetchone()[0]
    print(f"{name}: {n:,} rows")


=== PARTITION CHECK ===
daily_dec: 7,752,930 rows
daily_jan: 7,890,817 rows
daily_feb: 7,355,108 rows
daily_mar: 9,841,378 rows


## 1. Unit of analysis + time window

**One row in the analysis table = one pseudonymized content item for the March 2026 snapshot.**

The raw daily fact is at `report_date × client × content`. For this contract, those daily rows are
aggregated to content level.

Feature window: **2025-12-01 through 2026-02-28** (the 90 days immediately before March).

Outcome window: **2026-03-01 through 2026-03-31**.

The March outcome is not used to construct the pre-March features. This keeps the prediction
moment and the label window separate.

In [4]:
print("=== RAW FACT GRAIN ===")

# Compare total rows with distinct client/content/date keys.
grain_result = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date))
            AS distinct_grain_rows
    FROM {TABLES["daily_mar"]}
""").df()

total_rows = int(grain_result.loc[0, "total_rows"])
distinct_rows = int(grain_result.loc[0, "distinct_grain_rows"])
duplicate_rows = total_rows - distinct_rows

print(f"March daily rows: {total_rows:,}")
print(f"Distinct client/content/date keys: {distinct_rows:,}")
print(f"Duplicate daily-grain rows: {duplicate_rows:,}")
print("Expected: 0")

print("\n=== MARCH DATE WINDOW ===")

print(
    con.sql(f"""
        SELECT
            MIN(report_date) AS min_date,
            MAX(report_date) AS max_date,
            COUNT(*) AS daily_rows,
            COUNT(DISTINCT content_hash_id) AS content_items,
            COUNT(DISTINCT client_hash_id) AS clients
        FROM {TABLES["daily_mar"]}
    """).df().to_string(index=False)
)

print("\n=== PRE-MARCH FEATURE WINDOW ===")

print(
    con.sql(f"""
        SELECT
            MIN(min_date) AS min_date,
            MAX(max_date) AS max_date
        FROM (
            SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date
            FROM {TABLES["daily_dec"]}

            UNION ALL

            SELECT MIN(report_date), MAX(report_date)
            FROM {TABLES["daily_jan"]}

            UNION ALL

            SELECT MIN(report_date), MAX(report_date)
            FROM {TABLES["daily_feb"]}
        )
    """).df().to_string(index=False)
)


=== RAW FACT GRAIN ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March daily rows: 9,841,378
Distinct client/content/date keys: 9,841,378
Duplicate daily-grain rows: 0
Expected: 0

=== MARCH DATE WINDOW ===
  min_date   max_date  daily_rows  content_items  clients
2026-03-01 2026-03-31     9841378         331437       55

=== PRE-MARCH FEATURE WINDOW ===
  min_date   max_date
2025-12-01 2026-02-28


## 2. Fields: feature / label / context / excluded

I classify only fields actually used by this contract. The rule is simple:

- **Feature:** known before the March prediction moment.
- **Label:** the March outcome being predicted, or a field used to construct it.
- **Context:** identifiers used for grouping or joins, never model inputs.
- **Excluded:** fields deliberately kept out because they would leak the outcome or are not needed
  for this analysis.

In [5]:
print("=== FIELD CLASSIFICATION ===")

classification = {
    "Features": [
        "pre_march_impressions_90d",
        "pre_march_clicks_90d",
        "pre_march_avg_position",
        "pre_march_ctr",
    ],
    "Label / label source": [
        "march_impressions",
        "february_impressions",
        "trend_pct_march_vs_february",
        "trend_direction_march",
        "is_declining_label",
    ],
    "Context": [
        "content_hash_id — pseudonymous content identifier; grouping/joining only",
        "client_hash_id — pseudonymous client identifier; grouped client holdout only",
        "report_date — used to construct windows, not a model feature",
    ],
    "Excluded": [
        "March clicks / March position — measured during the outcome month",
        "Any March traffic metric used directly as a feature — future information at prediction time",
        "Raw IDs as model inputs — would allow memorisation rather than generalisation",
        "Final-month warehouse _sample — sealed outcome month, not used for this mid-panel contract",
    ],
}

for bucket, fields in classification.items():
    print(f"\n{bucket}:")
    for field in fields:
        print(" -", field)

=== FIELD CLASSIFICATION ===

Features:
 - pre_march_impressions_90d
 - pre_march_clicks_90d
 - pre_march_avg_position
 - pre_march_ctr

Label / label source:
 - march_impressions
 - february_impressions
 - trend_pct_march_vs_february
 - trend_direction_march
 - is_declining_label

Context:
 - content_hash_id — pseudonymous content identifier; grouping/joining only
 - client_hash_id — pseudonymous client identifier; grouped client holdout only
 - report_date — used to construct windows, not a model feature

Excluded:
 - March clicks / March position — measured during the outcome month
 - Any March traffic metric used directly as a feature — future information at prediction time
 - Raw IDs as model inputs — would allow memorisation rather than generalisation
 - Final-month warehouse _sample — sealed outcome month, not used for this mid-panel contract


## 3. Build the contract table and verify it

The code below creates the actual analysis grain. The monthly warehouse scans are done separately so the large remote Hugging Face reads remain reliable in a local VS Code environment. It uses only December-February activity for
the feature vector and March/February impressions for the outcome.

A content item must have a positive pre-March impression history to be included in the measurable
analysis population. This is a scope limitation, not a statement that zero-impression content
does not exist in the wider warehouse.

In [6]:
print("=== BUILDING ANALYSIS TABLE ===")
print("Aggregating each month separately so the large Hugging Face scans stay manageable.")

def aggregate_month(table_ref, month_name):
    print(f"Reading {month_name}...")
    result = con.sql(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            AVG(NULLIF(gsc_avg_position, 0)) AS avg_position
        FROM {table_ref}
        GROUP BY 1, 2
    """).df()
    print(f"{month_name}: {len(result):,} client/content rows")
    return result

dec = aggregate_month(TABLES["daily_dec"], "December")
jan = aggregate_month(TABLES["daily_jan"], "January")
feb = aggregate_month(TABLES["daily_feb"], "February")
mar = aggregate_month(TABLES["daily_mar"], "March")

pre_march = (
    pd.concat([dec, jan, feb], ignore_index=True)
      .groupby(["client_hash_id", "content_hash_id"], as_index=False)
      .agg(
          pre_march_impressions_90d=("impressions", "sum"),
          pre_march_clicks_90d=("clicks", "sum"),
          pre_march_avg_position=("avg_position", "mean"),
      )
)

feb = feb.rename(columns={"impressions": "february_impressions"})[
    ["client_hash_id", "content_hash_id", "february_impressions"]
]

mar = mar.rename(columns={"impressions": "march_impressions"})[
    ["client_hash_id", "content_hash_id", "march_impressions"]
]

analysis = (
    pre_march
    .merge(feb, on=["client_hash_id", "content_hash_id"], how="left")
    .merge(mar, on=["client_hash_id", "content_hash_id"], how="left")
)

analysis["february_impressions"] = analysis["february_impressions"].fillna(0)
analysis["march_impressions"] = analysis["march_impressions"].fillna(0)

analysis["pre_march_ctr"] = (
    100.0 * analysis["pre_march_clicks_90d"]
    / analysis["pre_march_impressions_90d"].replace(0, pd.NA)
)

analysis["trend_pct_march_vs_february"] = (
    100.0
    * (analysis["march_impressions"] - analysis["february_impressions"])
    / analysis["february_impressions"].replace(0, pd.NA)
)

feb_zero = analysis["february_impressions"].eq(0)
mar_positive = analysis["march_impressions"].gt(0)

analysis["trend_direction_march"] = "stable"
analysis.loc[feb_zero & mar_positive, "trend_direction_march"] = "new"

analysis.loc[
    ~feb_zero
    & (analysis["march_impressions"] > 1.20 * analysis["february_impressions"]),
    "trend_direction_march",
] = "up"

analysis.loc[
    ~feb_zero
    & (analysis["march_impressions"] < 0.80 * analysis["february_impressions"]),
    "trend_direction_march",
] = "down"

analysis["is_declining_label"] = (
    ~feb_zero
    & (analysis["march_impressions"] < 0.80 * analysis["february_impressions"])
).astype(int)

analysis = analysis[
    analysis["pre_march_impressions_90d"] > 0
].reset_index(drop=True)

print(f"Analysis rows: {len(analysis):,}")
print(f"Unique content items: {analysis['content_hash_id'].nunique():,}")
print(f"Unique clients: {analysis['client_hash_id'].nunique():,}")
print(
    f"Duplicate content rows: "
    f"{analysis['content_hash_id'].duplicated().sum():,}"
)


=== BUILDING ANALYSIS TABLE ===
Aggregating each month separately so the large Hugging Face scans stay manageable.
Reading December...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

December: 254,621 client/content rows
Reading January...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

January: 261,984 client/content rows
Reading February...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February: 321,546 client/content rows
Reading March...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March: 331,437 client/content rows
Analysis rows: 172,684
Unique content items: 172,684
Unique clients: 50
Duplicate content rows: 0


## 4. Missing values and window checks

Missingness is reported rather than silently filled. In particular, an absent position is not
treated as rank zero.

The warehouse documentation also warns that GA4 availability is not equivalent to a zero value;
the access flag must be considered when GA4 fields are used. This contract therefore keeps the
core GSC feature set small and explicit.

In [7]:
print("=== ANALYSIS MISSINGNESS ===")

missing = (
    analysis.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

print(missing[missing > 0].round(2).to_string())

print("\n=== WINDOW CHECKS ===")
print(
    "Pre-March impressions min/max:",
    int(analysis["pre_march_impressions_90d"].min()),
    int(analysis["pre_march_impressions_90d"].max()),
)
print(
    "Pre-March clicks min/max:",
    int(analysis["pre_march_clicks_90d"].min()),
    int(analysis["pre_march_clicks_90d"].max()),
)

print("\n=== LABEL DISTRIBUTION ===")
print(analysis["trend_direction_march"].value_counts(dropna=False).to_string())
print(
    "\nDeclining label rate:",
    f"{analysis['is_declining_label'].mean():.1%}",
)


=== ANALYSIS MISSINGNESS ===
trend_pct_march_vs_february    11.08
pre_march_avg_position          0.77

=== WINDOW CHECKS ===
Pre-March impressions min/max: 1 480045
Pre-March clicks min/max: 0 8092

=== LABEL DISTRIBUTION ===
trend_direction_march
up        78376
down      46146
stable    42678
new        5484

Declining label rate: 26.7%


## 5. Grain and client checks

The analysis table should have one row per content item. The raw daily table has a finer grain:
`report_date × client_hash_id × content_hash_id`.

Client IDs are retained only so that later validation can hold out complete clients rather than
randomly mixing pages from the same client between train and test.

In [8]:
print("=== ANALYSIS GRAIN CHECK ===")

dupes = (
    analysis.groupby("content_hash_id")
    .size()
    .reset_index(name="n")
    .query("n > 1")
)

print("Duplicate content rows:", len(dupes))
print("Expected: 0")

print("\n=== CLIENT DISTRIBUTION ===")
print(
    analysis.groupby("client_hash_id")
    .size()
    .describe()
    .to_string()
)

print("\nClient count:", analysis["client_hash_id"].nunique())

=== ANALYSIS GRAIN CHECK ===
Duplicate content rows: 0
Expected: 0

=== CLIENT DISTRIBUTION ===
count       50.00000
mean      3453.68000
std       5795.80129
min          1.00000
25%        122.75000
50%       1067.00000
75%       3287.50000
max      24537.00000

Client count: 50


## 6. Leakage check

The main leakage boundary is March.

The features are constructed entirely from December-February data. The March impressions are
used only to construct the label.

The following check confirms that no March-derived metric is present in the feature columns.

In [9]:
feature_cols = [
    "pre_march_impressions_90d",
    "pre_march_clicks_90d",
    "pre_march_avg_position",
    "pre_march_ctr",
]

label_cols = [
    "march_impressions",
    "february_impressions",
    "trend_pct_march_vs_february",
    "trend_direction_march",
    "is_declining_label",
]

overlap = set(feature_cols) & set(label_cols)

print("Feature/label column overlap:", overlap)
print("Expected: empty set")

assert not overlap

assert "march_impressions" not in feature_cols
assert "trend_direction_march" not in feature_cols
assert "trend_pct_march_vs_february" not in feature_cols

print("Leakage checks passed.")

Feature/label column overlap: set()
Expected: empty set
Leakage checks passed.


## 7. Data limits

This contract supports a directional, decision-support analysis. It does not establish causality.

The main limits are:

1. **No causal claim.** The warehouse is observational. A relationship between a feature and
   March performance is not evidence that changing that feature caused the outcome.

2. **Mid-panel snapshot only.** March 2026 is one historical evaluation window. A later month
   can behave differently.

3. **Unbalanced client history.** Clients have different amounts of historical data. Client
   coverage must be checked before comparing groups.

4. **Zero-impression content is outside this analysis population.** The contract filters to
   content with positive pre-March impressions, so it cannot answer which never-visible content
   will receive impressions.

5. **Position zero is not rank zero.** A zero GSC position value means there is no usable position
   measurement and must not be interpreted as the best possible rank.

6. **Window choice affects the result.** The feature window and March label window are deliberately
   separated. Reusing March data as a feature would invalidate the contract.

7. **Client IDs are pseudonyms.** They are useful for grouped validation but cannot be interpreted
   as real client identities.

8. **AI/GA4 fields require availability flags.** If those fields are added later, their availability
   flags must be handled explicitly rather than treating missing or zero-filled values as ordinary
   observations.

9. **Query-level 90-day data has its own window.** If it is added as a feature, its window must be
   aligned with the prediction moment first. Otherwise it can leak the outcome period.

## 8. Final self-check

- Every section has a written contract statement and a supporting query/check.
- The raw warehouse grain and the derived analysis grain are stated separately.
- March 2026 is used as a mid-panel snapshot rather than the sealed final-month sample.
- IDs are context only.
- March outcome fields are not model features.
- Missingness is measured rather than silently converted to zero.
- No client names, domains, URLs, private queries, or raw warehouse data are written to the repo.
- The Hugging Face token is not stored in this notebook.

In [10]:
print("SELF-CHECK")

checks = {
    "analysis has one row per content": analysis["content_hash_id"].is_unique,
    "no feature/label overlap": len(overlap) == 0,
    "March is label period": "march_impressions" in label_cols,
    "no duplicate raw daily grain": duplicate_rows == 0,
}

for name, ok in checks.items():
    print(f"{name}: {'PASS' if ok else 'CHECK'}")

print("\nContract complete. Run the notebook top-to-bottom before submission.")


SELF-CHECK
analysis has one row per content: PASS
no feature/label overlap: PASS
March is label period: PASS
no duplicate raw daily grain: PASS

Contract complete. Run the notebook top-to-bottom before submission.
